In [3]:
# File: 3_train_timm_with_stats.py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import joblib
import cv2
import os
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import timm

# --- 1. Configuration (Unchanged) ---
TRAIN_DIR = r"D:\UPenn\CIS-5190\project\data\train\all" # <-- Using 'internal'
VAL_DIR = r"D:\UPenn\CIS-5190\project\data\validation" # <-- Using 'val'
STATS_PATH_DELETE = r"D:\UPenn\CIS-5190\project\models\gps_stats.pt" # <-- We won't use this file for submission
MODEL_SAVE_PATH = "models/dinov3_base.pth" # <-- New save path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 64 # As in your script
EPOCHS = 50
LEARNING_RATE = 1e-3
cv2.setNumThreads(0)

# --- 2. Data Augmentation (Unchanged, "No Augmentation") ---
train_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# --- 3. Custom PyTorch Dataset (Unchanged) ---
class CampusDataset(Dataset):
    def __init__(self, df, image_dir, gps_mean, gps_std, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.gps_mean = gps_mean.to(torch.float32)
        self.gps_std = gps_std.to(torch.float32)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_dir, row['file_name'])
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
        gps_coords = row[['Latitude', 'Longitude']].values.astype(np.float32)
        label_tensor = torch.from_numpy(gps_coords)
        scaled_label = (label_tensor - self.gps_mean) / self.gps_std
        return image, scaled_label

# --- 4. Model Definition (MODIFIED TO HOLD STATS) ---
class GpsTimmModel(nn.Module):
    def __init__(self, model_name, gps_mean, gps_std, n_outputs=2):
        super(GpsTimmModel, self).__init__()
        
        self.backbone = timm.create_model(
            model_name, pretrained=True, num_classes=0, global_pool=''
        )
        num_features = self.backbone.num_features
        self.pooling = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.GELU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, n_outputs)
        )
        
        # --- THIS IS THE KEY ---
        # Register gps_mean and gps_std as "buffers".
        # This makes them part of the model's state_dict,
        # so they get saved in the .pth file.
        self.register_buffer("gps_mean", gps_mean)
        self.register_buffer("gps_std", gps_std)
        # --- END OF KEY ---

    def forward(self, x):
        feature_maps = self.backbone(x)
        pooled_features = self.pooling(feature_maps)
        flattened_features = torch.flatten(pooled_features, 1)
        return self.head(flattened_features)

def build_timm_model(gps_mean, gps_std): # <-- Now accepts stats
    model_name = "convnext_base.dinov3_lvd1689m"
    # Pass stats to the constructor
    model = GpsTimmModel(model_name, gps_mean, gps_std)
    for param in model.backbone.parameters():
        param.requires_grad = False
    return model
# --- End of Model Definition ---

# --- 5. Training & Validation Functions
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    for images, labels in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(dataloader.dataset)

def validate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item() * images.size(0)
    return total_loss / len(dataloader.dataset)

# --- 6. Main Script (MODIFIED) ---
if __name__ == '__main__':
    print(f"Using device: {DEVICE}")

    train_df = pd.read_csv(os.path.join(TRAIN_DIR, "metadata.csv"))
    val_df = pd.read_csv(os.path.join(VAL_DIR, "metadata.csv"))
    
    # --- Calculate and save mean/std ---
    print("Calculating GPS mean/std from training data...")
    lat_mean = train_df['Latitude'].mean()
    lat_std = train_df['Latitude'].std()
    lon_mean = train_df['Longitude'].mean()
    lon_std = train_df['Longitude'].std()
    
    gps_mean = torch.tensor([lat_mean, lon_mean], dtype=torch.float32)
    gps_std = torch.tensor([lat_std, lon_std], dtype=torch.float32)
    
    # --- BUILD MODEL *AFTER* CALCULATING STATS ---
    model = build_timm_model(gps_mean, gps_std).to(DEVICE)
    print("Model built and GPS stats have been embedded.")
    
    # --- Create Datasets (Now pass tensors) ---
    train_dataset = CampusDataset(df=train_df, image_dir=TRAIN_DIR, 
                                  gps_mean=gps_mean, gps_std=gps_std, 
                                  transform=train_transform)
    val_dataset = CampusDataset(df=val_df, image_dir=VAL_DIR, 
                                gps_mean=gps_mean, gps_std=gps_std, 
                                transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print("DataLoaders created.")
    
    loss_fn = nn.HuberLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE) # Train all params
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.1)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    patience = 5
    
    # --- STAGE 1 (Training everything) ---
    # We will just use ONE training stage, as your provided script did.
    # The two-stage loop can be added back, but this simplifies things.
    for epoch in range(EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        val_loss = validate(model, val_loader, loss_fn, DEVICE)
        print(f"Epoch {epoch+1}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # Save the full model state_dict, which now INCLUDES the buffers
            torch.save(model.state_dict(), MODEL_SAVE_PATH) 
            print(f"Model saved to {MODEL_SAVE_PATH}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break
        # --- STAGE 2: FINE-TUNING (Train everything) ---
    print("\n--- Stage 1 Complete. Loading best head weights. ---")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    print("\n--- Starting Stage 2: Fine-Tuning Full Model ---")
    
    for param in model.parameters():
        param.requires_grad = True
        
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE / 100) # Use a very low LR
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.1)
    best_val_loss = float('inf') 
    epochs_no_improve = 0
    
    for epoch in range(EPOCHS): 
        print(f"\n--- Fine-Tuning Epoch {epoch+1}/{EPOCHS} ---")
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        val_loss = validate(model, val_loader, loss_fn, DEVICE)
        print(f"Epoch {epoch+1}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"Model saved to {MODEL_SAVE_PATH}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered for Stage 2.")
                break
                
    print("\nTraining complete.")

KeyboardInterrupt: 

In [3]:
    # ... (Stage 2) ...
# File: 3_train_timm_with_stats.py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import joblib
import cv2
import os
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import timm
# --- 1. Configuration (Unchanged) ---
TRAIN_DIR = r"D:\UPenn\CIS-5190\project\data\train\all" # <-- Using 'internal'
VAL_DIR = r"D:\UPenn\CIS-5190\project\data\validation" # <-- Using 'val'
STATS_PATH_DELETE = r"D:\UPenn\CIS-5190\project\models\gps_stats.pt" # <-- We won't use this file for submission
MODEL_SAVE_PATH = "models/dinov3_base.pth" # <-- New save path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32 # As in your script
EPOCHS = 50
LEARNING_RATE = 1e-3
cv2.setNumThreads(0)

# --- 2. Data Augmentation (Unchanged, "No Augmentation") ---
train_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# --- 3. Custom PyTorch Dataset (Unchanged) ---
class CampusDataset(Dataset):
    def __init__(self, df, image_dir, gps_mean, gps_std, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.gps_mean = gps_mean.to(torch.float32)
        self.gps_std = gps_std.to(torch.float32)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_dir, row['file_name'])
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
        gps_coords = row[['Latitude', 'Longitude']].values.astype(np.float32)
        label_tensor = torch.from_numpy(gps_coords)
        scaled_label = (label_tensor - self.gps_mean) / self.gps_std
        return image, scaled_label

# --- 4. Model Definition (MODIFIED TO HOLD STATS) ---
class GpsTimmModel(nn.Module):
    def __init__(self, model_name, gps_mean, gps_std, n_outputs=2):
        super(GpsTimmModel, self).__init__()
        
        self.backbone = timm.create_model(
            model_name, pretrained=True, num_classes=0, global_pool=''
        )
        num_features = self.backbone.num_features
        self.pooling = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.GELU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, n_outputs)
        )
        
        # --- THIS IS THE KEY ---
        # Register gps_mean and gps_std as "buffers".
        # This makes them part of the model's state_dict,
        # so they get saved in the .pth file.
        self.register_buffer("gps_mean", gps_mean)
        self.register_buffer("gps_std", gps_std)
        # --- END OF KEY ---

    def forward(self, x):
        feature_maps = self.backbone(x)
        pooled_features = self.pooling(feature_maps)
        flattened_features = torch.flatten(pooled_features, 1)
        return self.head(flattened_features)

def build_timm_model(gps_mean, gps_std): # <-- Now accepts stats
    model_name = "convnext_base.dinov3_lvd1689m"
    # Pass stats to the constructor
    model = GpsTimmModel(model_name, gps_mean, gps_std)
    for param in model.backbone.parameters():
        param.requires_grad = False
    return model
# --- End of Model Definition ---

# --- 5. Training & Validation Functions
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    for images, labels in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(dataloader.dataset)

def validate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item() * images.size(0)
    return total_loss / len(dataloader.dataset)

# --- 6. Main Script (MODIFIED) ---
if __name__ == '__main__':
    print(f"Using device: {DEVICE}")

    train_df = pd.read_csv(os.path.join(TRAIN_DIR, "metadata.csv"))
    val_df = pd.read_csv(os.path.join(VAL_DIR, "metadata.csv"))
    
    # --- Calculate and save mean/std ---
    print("Calculating GPS mean/std from training data...")
    lat_mean = train_df['Latitude'].mean()
    lat_std = train_df['Latitude'].std()
    lon_mean = train_df['Longitude'].mean()
    lon_std = train_df['Longitude'].std()
    
    gps_mean = torch.tensor([lat_mean, lon_mean], dtype=torch.float32)
    gps_std = torch.tensor([lat_std, lon_std], dtype=torch.float32)
    
    # --- BUILD MODEL *AFTER* CALCULATING STATS ---
    model = build_timm_model(gps_mean, gps_std).to(DEVICE)
    print("Model built and GPS stats have been embedded.")
    
    # --- Create Datasets (Now pass tensors) ---
    train_dataset = CampusDataset(df=train_df, image_dir=TRAIN_DIR, 
                                  gps_mean=gps_mean, gps_std=gps_std, 
                                  transform=train_transform)
    val_dataset = CampusDataset(df=val_df, image_dir=VAL_DIR, 
                                gps_mean=gps_mean, gps_std=gps_std, 
                                transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print("DataLoaders created.")
    
    loss_fn = nn.HuberLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE) # Train all params
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.1)

    print("\n--- Stage 1 Complete. Loading best head weights. ---")
    # Load the best weights
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    
    # --- THE FIX: Explicitly set requires_grad for ALL parameters ---
    # We iterate through named_parameters to be safe
    print("Unfreezing all parameters for fine-tuning...")
    for name, param in model.named_parameters():
        param.requires_grad = True
    
    # Verify that parameters are actually trainable
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters for Stage 2: {trainable_params}")

    # Re-create the optimizer fresh
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE / 100)
    
    # Reset scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.1)
    
    print("\n--- Starting Stage 2: Fine-Tuning Full Model ---")
    
    best_val_loss = float('inf') 
    epochs_no_improve = 0
    patience = 5
    for epoch in range(EPOCHS): 
        print(f"\n--- Fine-Tuning Epoch {epoch+1}/{EPOCHS} ---")
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        val_loss = validate(model, val_loader, loss_fn, DEVICE)
        print(f"Epoch {epoch+1}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"Model saved to {MODEL_SAVE_PATH}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered for Stage 2.")
                break
    print("\nTraining complete.")

Using device: cuda
Calculating GPS mean/std from training data...
Model built and GPS stats have been embedded.
DataLoaders created.

--- Stage 1 Complete. Loading best head weights. ---
Unfreezing all parameters for fine-tuning...
Total trainable parameters for Stage 2: 88092290

--- Starting Stage 2: Fine-Tuning Full Model ---

--- Fine-Tuning Epoch 1/50 ---


Validating: 100%|██████████| 45/45 [00:47<00:00,  1.05s/it]


Epoch 1: Train Loss = 0.001421, Val Loss = 0.006093
Model saved to models/dinov3_base.pth

--- Fine-Tuning Epoch 2/50 ---


Validating: 100%|██████████| 45/45 [00:48<00:00,  1.08s/it]


Epoch 2: Train Loss = 0.001654, Val Loss = 0.006161

--- Fine-Tuning Epoch 3/50 ---


...

Epoch 17: Train Loss = 0.000967, Val Loss = 0.004970

--- Fine-Tuning Epoch 18/50 ---


Validating: 100%|██████████| 45/45 [00:48<00:00,  1.08s/it]


Epoch 18: Train Loss = 0.000966, Val Loss = 0.004936

--- Fine-Tuning Epoch 19/50 ---


Validating: 100%|██████████| 45/45 [00:49<00:00,  1.09s/it]

Epoch 19: Train Loss = 0.000943, Val Loss = 0.004967
Early stopping triggered for Stage 2.

Training complete.


In [2]:
    # ... (Stage 3) ...
    print("\n--- Stage 2 Complete. Loading best head weights. ---")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    print("\n--- Starting Stage 3: Ultra Fine-Tuning Full Model ---")
    
    for param in model.parameters():
        param.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE / 1000)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.1)
    best_val_loss = float('inf') 
    epochs_no_improve = 0
    for epoch in range(EPOCHS): 
        print(f"\n--- Fine-Tuning Epoch {epoch+1}/{EPOCHS} ---")
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        val_loss = validate(model, val_loader, loss_fn, DEVICE)
        print(f"Epoch {epoch+1}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"Model saved to {MODEL_SAVE_PATH}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered for Stage 2.")
                break
    print("\nTraining complete.")


--- Stage 2 Complete. Loading best head weights. ---

--- Starting Stage 3: Ultra Fine-Tuning Full Model ---

--- Fine-Tuning Epoch 1/50 ---


Validating: 100%|██████████| 45/45 [00:46<00:00,  1.03s/it]


Epoch 1: Train Loss = 0.001084, Val Loss = 0.006348
Model saved to models/dinov3_base.pth

--- Fine-Tuning Epoch 2/50 ---


Validating: 100%|██████████| 45/45 [00:47<00:00,  1.05s/it]


Epoch 2: Train Loss = 0.001072, Val Loss = 0.006411

--- Fine-Tuning Epoch 3/50 ---


...

Epoch 9: Train Loss = 0.000986, Val Loss = 0.006291

--- Fine-Tuning Epoch 10/50 ---


Validating: 100%|██████████| 45/45 [00:46<00:00,  1.04s/it]


Epoch 10: Train Loss = 0.000996, Val Loss = 0.006429

--- Fine-Tuning Epoch 11/50 ---


Validating: 100%|██████████| 45/45 [00:46<00:00,  1.03s/it]

Epoch 11: Train Loss = 0.000967, Val Loss = 0.006317
Early stopping triggered for Stage 2.

Training complete.
